## XGBoost v1

- No data preprocessing
- No model tweaking/optimization, only a basic implementation of XGBoost
- Baseline XGBoost model

In [1]:
from dotenv import load_dotenv
import kagglehub
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import torch
from xgboost import XGBClassifier

if torch.cuda.is_available():
    device="cuda"
else:
    device="cpu"
device

'cuda'

In [2]:
load_dotenv()

if not Path("../Data/.complete/competitions/playground-series-s6e6/bundle.complete").is_file():
    kagglehub.competition_download("playground-series-s6e6", output_dir="Data")
    print("Data downloaded")
else:
    print("Data not downloaded, already exists")

Data not downloaded, already exists


In [3]:
train = pd.read_csv("../Data/train.csv", index_col="id")
train.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
id,,,,,,,,,,,
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [4]:
train.dtypes

alpha                float64
delta                float64
u                    float64
g                    float64
r                    float64
i                    float64
z                    float64
redshift             float64
spectral_type         object
galaxy_population     object
class                 object
dtype: object

In [5]:
X = train.drop(columns=["class"])
categorical_cols = ["spectral_type", "galaxy_population"]
for col in categorical_cols:
    X[col] = X[col].astype("category")

le = LabelEncoder()
y = le.fit_transform(train["class"])

X_train, X_val, y_train, y_val = train_test_split(
    X, 
    y, 
    test_size=0.2,
    random_state=42, 
    stratify=y
)

In [6]:
model = XGBClassifier(
    device=device,
    random_state=42,
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=5,
    early_stopping_rounds=100,
    enable_categorical=True,
    verbosity=0
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)]
)

preds = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, preds))
print(classification_report(y_val, preds, target_names=le.classes_))

[0]	validation_0-mlogloss:0.82304
[1]	validation_0-mlogloss:0.77355
[2]	validation_0-mlogloss:0.72976
[3]	validation_0-mlogloss:0.69068
[4]	validation_0-mlogloss:0.65540
[5]	validation_0-mlogloss:0.62336
[6]	validation_0-mlogloss:0.59424


sh: 1: nvidia-smi: not found


[7]	validation_0-mlogloss:0.56737
[8]	validation_0-mlogloss:0.54269
[9]	validation_0-mlogloss:0.51991
[10]	validation_0-mlogloss:0.49863
[11]	validation_0-mlogloss:0.47904
[12]	validation_0-mlogloss:0.46065
[13]	validation_0-mlogloss:0.44358
[14]	validation_0-mlogloss:0.42750
[15]	validation_0-mlogloss:0.41249
[16]	validation_0-mlogloss:0.39845
[17]	validation_0-mlogloss:0.38534
[18]	validation_0-mlogloss:0.37304
[19]	validation_0-mlogloss:0.36141
[20]	validation_0-mlogloss:0.35055
[21]	validation_0-mlogloss:0.34016
[22]	validation_0-mlogloss:0.33048
[23]	validation_0-mlogloss:0.32133
[24]	validation_0-mlogloss:0.31271
[25]	validation_0-mlogloss:0.30456
[26]	validation_0-mlogloss:0.29683
[27]	validation_0-mlogloss:0.28951
[28]	validation_0-mlogloss:0.28256
[29]	validation_0-mlogloss:0.27596
[30]	validation_0-mlogloss:0.26971
[31]	validation_0-mlogloss:0.26371
[32]	validation_0-mlogloss:0.25810
[33]	validation_0-mlogloss:0.25279
[34]	validation_0-mlogloss:0.24767
[35]	validation_0-mlogl

In [7]:
test = pd.read_csv("../Data/test.csv", index_col="id")

for col in categorical_cols:
    test[col] = test[col].astype("category")

test_preds = model.predict(test)
test_pred_labels = le.inverse_transform(test_preds)

submission = pd.DataFrame({
    "id": test.index,
    "class": test_pred_labels
})

submission.to_csv("../Data/submission_xgboost_v1.csv", index=False)

submission.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
